# Diffusion World Model on Kaggle — Push-T (`swm/PushT-v1`)

Trains and evaluates the **SD-VAE (4x28x28) + Rectified Flow / Diffusion Forcing DiT** world model on **Push-T**.

> **Kaggle Settings Required:**
> - **Accelerator:** GPU (T4 x2 or P100 16 GB)
> - **Internet:** ON

In [ ]:
# === 1. User Configuration ===
GITHUB_REPO_URL = "https://github.com/hanriman/diffusion-wm.git"
BRANCH = "main"

ENV_NAME = "pusht"
SOLVER = "gd"        # "gd" (GradientSolver) or "cem" (CEMSolver)
SEED = 42

PRECOMPUTE_VAE_LATENTS = False  # False avoids ~15-30GB disk usage on Push-T 224x224
MAX_EPOCHS = 5
BATCH_SIZE = 32
LIMIT_TRAIN_BATCHES = 2000  # batches per epoch (set None for full dataset)
LIMIT_VAL_BATCHES = 100     # validation batches per epoch (set None for full val set)
TOTAL_TRANSITIONS = None    # optional dataset transition cap (e.g. 200000) or None
NUM_EVAL = 10               # set 50 for full benchmark eval
EVAL_NUM_SAMPLES = 64 if SOLVER == "gd" else 128

REPO_DIR = "/kaggle/working/diffusion-wm"
DATA_DIR = "/tmp/hjepa_data"
CKPT_DIR = f"/kaggle/working/ckpts/{ENV_NAME}/{ENV_NAME}_dwm/seed{SEED}"
KAGGLE_INPUT_DATA_DIR = ""  # Option A: leave empty to auto-download from HuggingFace

In [ ]:
%%bash
nvidia-smi
df -h /kaggle/working /tmp
apt-get update -qq && apt-get install -y -qq zstd zip > /dev/null || true
pip install -q uv
which uv && uv --version

In [ ]:
import os
import subprocess

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["HJEPA_HOME"] = DATA_DIR
os.environ["MUJOCO_GL"] = "egl"
os.environ["MPLBACKEND"] = "Agg"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

%cd {REPO_DIR}

In [ ]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
uv venv --python 3.11
uv pip install -e ".[env]" pytest
uv run python -c "import torch; print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')"

### 2. Download Evaluation Tasks & PUSHT Dataset (Option A)
- Downloads the 50 fixed evaluation tasks into `diffusion_wm/assets/eval_trajs/`.
- Downloads `pusht` HDF5 train/val datasets into `DATA_DIR` (`/tmp/hjepa_data`).

In [ ]:
import glob
import subprocess

subprocess.run(["bash", "scripts/download_eval_trajs.sh"], check=True)

if KAGGLE_INPUT_DATA_DIR and os.path.isdir(KAGGLE_INPUT_DATA_DIR):
    for h5_file in glob.glob(os.path.join(KAGGLE_INPUT_DATA_DIR, "*.h5")):
        dst = os.path.join(DATA_DIR, os.path.basename(h5_file))
        if not os.path.exists(dst):
            os.symlink(h5_file, dst)
    print("Linked HDF5 files from Kaggle Input:", os.listdir(DATA_DIR))
else:
    subprocess.run(["bash", "scripts/download_datasets.sh", ENV_NAME], check=True)

### 3. (Optional) Precompute SD-VAE Latents
Skipped when `PRECOMPUTE_VAE_LATENTS = False`.

In [ ]:
if PRECOMPUTE_VAE_LATENTS:
    h5_files = ["pusht_expert_train.h5", "pusht_expert_val.h5"]
    for fname in h5_files:
        fpath = os.path.join(DATA_DIR, fname)
        subprocess.run([
            "uv", "run", "python", "scripts/precompute_vae_latents.py",
            fpath, "--batch-size", "64"
        ], check=True)

### 4. Train Diffusion World Model
Uses `trainer.precision=16-mixed` and `trainer.devices=1` for Kaggle T4/P100 compatibility, and saves checkpoints + `normalizer.pt` to `/kaggle/working/ckpts/...`.

In [ ]:
train_cfg = f"{ENV_NAME}_dwm"
cmd = [
    "uv", "run", "python", "-m", "diffusion_wm.train",
    "--config-name", train_cfg,
    f"seed={SEED}",
    f"cache_dir={DATA_DIR}",
    f"run_dir={CKPT_DIR}",
    "trainer.precision=16-mixed",
    "trainer.devices=1",
    f"trainer.max_epochs={MAX_EPOCHS}",
    f"loader.batch_size={BATCH_SIZE}",
    "loader.num_workers=2",
    "planning_eval.enabled=false",
]

if LIMIT_TRAIN_BATCHES is not None:
    cmd.append(f"trainer.limit_train_batches={LIMIT_TRAIN_BATCHES}")
if LIMIT_VAL_BATCHES is not None:
    cmd.append(f"trainer.limit_val_batches={LIMIT_VAL_BATCHES}")
if TOTAL_TRANSITIONS is not None:
    cmd.append(f"data.dataset.total_transitions={TOTAL_TRANSITIONS}")

if PRECOMPUTE_VAE_LATENTS:
    cmd.append("data.dataset.keys_to_load=[vae_latents,action,proprio,state]")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

### 5. Evaluate Closed-Loop MPC Planning
Runs `diffusion_wm.eval` using the trained checkpoint and prints `metrics.yaml`.

In [ ]:
ckpt_path = os.path.join(CKPT_DIR, f"{ENV_NAME}_dwm_object.ckpt")
eval_cfg = f"{ENV_NAME}_{SOLVER}"
eval_out_dir = f"eval_{SOLVER}"

eval_cmd = [
    "uv", "run", "python", "-m", "diffusion_wm.eval",
    "--config-name", eval_cfg,
    f"seed={SEED}",
    f"cache_dir={DATA_DIR}",
    f"policy={ckpt_path}",
    f"eval.num_eval={NUM_EVAL}",
    f"solver.num_samples={EVAL_NUM_SAMPLES}",
    f"output.dir={eval_out_dir}",
]
print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)

metrics_file = os.path.join(CKPT_DIR, eval_out_dir, "metrics.yaml")
with open(metrics_file) as f:
    print("\n=== Evaluation Metrics ===\n" + f.read())

### 6. Generate & Display Presentation Visuals
Generates training loss curves, open-loop **World Model Dreams** (`Ground Truth` vs `SD-VAE Recon` vs `Diffusion WM`), evaluation summary plots, and side-by-side **Closed-Loop MPC Rollout GIFs** for your presentation.

In [ ]:
import glob
from IPython.display import Image as IPyImage, display

viz_dir = os.path.join(CKPT_DIR, "visualizations")
viz_cmd = [
    "uv", "run", "python", "-m", "diffusion_wm.visualize",
    "--eval-config", eval_cfg,
    "--policy", ckpt_path,
    "--run-dir", CKPT_DIR,
    "--eval-dir", os.path.join(CKPT_DIR, eval_out_dir),
    "--out-dir", viz_dir,
    "--cache-dir", DATA_DIR,
    "--num-dreams", "2",
    "--num-videos", "2",
]
print("Running:", " ".join(viz_cmd))
subprocess.run(viz_cmd, check=True)

for img_path in sorted(glob.glob(os.path.join(viz_dir, "*.png")) + glob.glob(os.path.join(viz_dir, "*.gif"))):
    print("\n>>>", os.path.basename(img_path))
    display(IPyImage(filename=img_path))

### 7. Package Outputs for Download
Creates `/kaggle/working/pusht_results.zip` containing your checkpoints, normalizer, and evaluation metrics.

In [ ]:
zip_path = "/kaggle/working/pusht_results.zip"
subprocess.run(["zip", "-r", "-q", zip_path, f"/kaggle/working/ckpts/{ENV_NAME}"], check=True)
print("Ready to download:", zip_path, f"({os.path.getsize(zip_path) / (1024**2):.1f} MB)")